# Parquet

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/parquet/parquet.ipynb)

Apache Parquet is a columnar, typed file format for analytics and the default storage format for Spark, Trino, Athena, and lakehouse tables. The runnable cells use `pyarrow` on small synthetic data in a temporary directory.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyarrow": "pyarrow>=15"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import tempfile
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

WORKDIR = Path(tempfile.mkdtemp(prefix="parquet-demo-"))
COUNTRIES = ["DE", "FR", "IN", "US", "BR"]
DAYS = [date(2026, 10, 1) + timedelta(days=d) for d in range(4)]
ROWS = 40_000

orders = pa.table({
    "order_id": pa.array(range(ROWS), pa.int64()),
    "customer_id": pa.array([(i * 7919) % 5_000 for i in range(ROWS)], pa.int64()),
    "country": pa.array([COUNTRIES[i % len(COUNTRIES)] for i in range(ROWS)]),
    "order_date": pa.array([DAYS[i * len(DAYS) // ROWS] for i in range(ROWS)], pa.date32()),
    "amount": pa.array([Decimal(f"{(i * 37) % 50_000 / 100:.2f}") for i in range(ROWS)], pa.decimal128(12, 2)),
    "order_ts": pa.array(
        [datetime(2026, 10, 1, tzinfo=timezone.utc) + timedelta(seconds=i * 8) for i in range(ROWS)],
        pa.timestamp("us", tz="UTC"),
    ),
    "description": pa.array([f"order {i} free-text description " * 3 for i in range(ROWS)]),
})
print(orders.schema)
print("rows:", orders.num_rows, "workdir:", WORKDIR)

## What is Apache Parquet?

Apache Parquet is an open-source columnar file format designed for analytical workloads. Values are stored by column inside row groups, each column chunk carries statistics (min, max, null count), and a footer holds the schema and metadata. Query engines read only the columns and row groups they need, which is why Parquet scans are cheaper than row-oriented files.

Under the hood a Parquet file is a small metadata tree wrapped around blocks of column data. The file is split into row groups, a row group holds one column chunk per column, and each column chunk is split into pages. The footer is written last and records the schema, the offset and size of every row group and column chunk, the encodings used, and min/max/null-count statistics. A reader opens the footer first, then seeks only to the byte ranges it needs.

Pages are the compression and encoding unit. A page is either a dictionary page or a data page. Dictionary encoding replaces repeated values with small integer keys; run-length encoding (RLE) and bit-packing then compress those keys and the definition and repetition levels that mark nulls and nesting. Integers and timestamps often use delta or delta-binary-packed encoding. When a dictionary grows past the dictionary page limit, the writer falls back to plain encoding for the rest of the chunk.

Parquet is a file format, not a table format. It has no transactions, no row-level deletes, and no catalog. Iceberg, Delta Lake, and Hudi add those table semantics on top of Parquet files.

A file with several row groups shows the layout directly: the footer lists every row group and every column chunk, with byte sizes and statistics.

In [ ]:
orders_path = WORKDIR / "orders.parquet"
pq.write_table(orders, orders_path, row_group_size=10_000, compression="snappy")

metadata = pq.ParquetFile(orders_path).metadata
print("row groups:", metadata.num_row_groups, "columns:", metadata.num_columns, "footer bytes:", metadata.serialized_size)
first = metadata.row_group(0)
for i in range(first.num_columns):
    chunk = first.column(i)
    print(f"  {chunk.path_in_schema:12} {chunk.total_compressed_size:>8,} bytes  encodings={chunk.encodings}")

assert metadata.num_row_groups == 4
assert metadata.num_row_groups * metadata.num_columns == 28  # one column chunk per column per row group

## Why is Parquet better than CSV or JSON for analytics?

CSV and JSON are row-oriented text formats. They are human-readable but expensive: every query parses every row, every column, and often every byte. Parquet is binary, typed, columnar, compressed, and splittable, so engines like Spark, Hive, Athena, BigQuery external tables, Trino, and DuckDB scan far less data.

The cost model is bytes read, not rows read. A row-oriented file must be read end to end to answer any query because a row is stored contiguously. A columnar file stores each column separately, so an engine reads only the projected columns, decompresses only those chunks, and decodes only the values it needs. Typed columns also compress far better than text: integers and timestamps are bit-packed, repeated values are dictionary-encoded, and each column chooses its own codec.

| Aspect | JSON / CSV | Parquet |
|---|---|---|
| Storage size | 200 GB | 40-70 GB with Snappy, less with Zstd |
| Scan for 2 of 30 columns | reads every column | reads those 2 column chunks |
| Filter on one date | full scan, no skipping | skips row groups via min/max statistics |
| Schema | inferred per run | embedded in the footer |
| Splittable | often not (gzip CSV) | yes, splittable per row group |
| Nested fields | re-parsed as text | native struct/array/map types |

The same table as JSON lines and as Parquet, then a two-column read:

In [ ]:
import json

json_path = WORKDIR / "orders.jsonl"
with json_path.open("w") as fh:
    for row in orders.to_pylist():
        fh.write(json.dumps(row, default=str) + "\n")

json_bytes = json_path.stat().st_size
parquet_bytes = orders_path.stat().st_size
print(f"json lines: {json_bytes:>10,} bytes")
print(f"parquet:    {parquet_bytes:>10,} bytes  ({json_bytes / parquet_bytes:.1f}x smaller)")

# Column projection: only the two requested column chunks are decoded
projected = pq.read_table(orders_path, columns=["customer_id", "amount"])
projected_bytes = sum(
    metadata.row_group(rg).column(c).total_compressed_size
    for rg in range(metadata.num_row_groups)
    for c in (1, 4)
)
print(f"bytes in the 2 projected column chunks: {projected_bytes:,} of {parquet_bytes:,}")

assert parquet_bytes < json_bytes / 3
assert projected.column_names == ["customer_id", "amount"]
assert projected_bytes < parquet_bytes / 2

Common mistake: converting to Parquet but keeping a schema that defeats it. A table where analysts always select all 30 columns, or one wide `payload` string that holds the real data, gains little. Columnar formats pay off when queries touch a subset of narrow, typed columns.

## How do you write and read Parquet with PySpark?

A PySpark write is a distributed sequence of per-task Parquet writers. Each task buffers rows and flushes a row group once it reaches the configured row-group size (128 MB by default); at the end of the task it writes the footer and closes the file. The number of output files equals the number of tasks that have data, which is why a shuffle or `repartition` controls file count. Compression is applied per page and dictionary encoding is enabled by default.

A production-shaped Spark write (interviewers probe `mode`, `partitionBy`, compression, and file count):

```python
from pyspark.sql import functions as F

orders = (
    spark.read.json("s3://acme-raw/orders/")
    .withColumn("order_date", F.to_date("created_at"))
)

# One logical partition per day, deterministic writes, compressed
(
    orders.repartition("order_date")
    .write.mode("overwrite")
    .partitionBy("order_date")
    .option("compression", "snappy")
    .parquet("s3://acme-curated/orders/")
)

# Read it back and let the engine prune partitions and row groups
curated = spark.read.parquet("s3://acme-curated/orders/")
curated.filter(F.col("order_date") == "2026-10-04").groupBy("country").agg(F.sum("amount")).show()
```

The pyarrow equivalent with explicit layout knobs, runnable here:

In [ ]:
explicit_path = WORKDIR / "orders_explicit.parquet"
pq.write_table(
    orders,
    explicit_path,
    row_group_size=20_000,           # rows per row group, not bytes
    data_page_size=1 * 1024 * 1024,  # target page size before compression
    compression="zstd",
    compression_level=3,
    use_dictionary=["country"],      # dictionary-encode only the low-cardinality column
    write_statistics=True,
)
explicit = pq.ParquetFile(explicit_path).metadata
print("row groups:", explicit.num_row_groups, "compression:", explicit.row_group(0).column(0).compression)
assert explicit.num_row_groups == 2

+ `overwrite` replaces the whole target (and with `partitionBy`, only the touched partitions under dynamic partition overwrite).
+ `append` is cheap but creates duplicates if a task reruns, which is why idempotency belongs to the table format or to a partition-replace pattern.
+ `repartition(col)` yields one file per partition value; use `coalesce(n)` for the opposite (fewer files, no shuffle) when the data is already small.

## How do columnar storage and predicate pushdown make queries faster?

Parquet stores each column contiguously, so a scan is proportional to the columns selected, not the rows. On top of that, Parquet metadata enables three levels of skipping:

1. **Partition pruning** - the engine skips whole directories from the partition path (`ds=2026-10-03/`).
2. **Row-group skipping** - the footer stores per-column min/max and null counts, so a row group is skipped when the filter cannot match.
3. **Page/dictionary filtering** - finer-grained statistics and bloom filters can skip pages inside a row group.

The engine reads statistics in three stages. First it lists partition directories and drops those that cannot match. Then it opens each file footer and compares the predicate against the per-column-chunk min/max and null counts, skipping whole row groups. Then, if a page index is present (`write_page_index=True` in pyarrow, or a Parquet v2 writer), it can skip individual pages. Bloom filters add equality skipping on high-cardinality columns where broad min/max ranges are useless.

Below, the same data is written as a hive-partitioned dataset sorted by `order_ts`. The filter prunes partition directories first, then row groups by min/max statistics.

In [ ]:
dataset_dir = WORKDIR / "orders_by_day"
day_partitioning = ds.partitioning(pa.schema([("order_date", pa.date32())]), flavor="hive")
ds.write_dataset(
    orders,
    dataset_dir,
    format="parquet",
    partitioning=day_partitioning,
    file_options=ds.ParquetFileFormat().make_write_options(compression="snappy"),
    max_rows_per_group=2_000,
    min_rows_per_group=2_000,
)
print(sorted(p.relative_to(dataset_dir).as_posix() for p in dataset_dir.rglob("*.parquet")))

dataset = ds.dataset(dataset_dir, format="parquet", partitioning=day_partitioning)
target_day = DAYS[2]
day_filter = ds.field("order_date") == pa.scalar(target_day, pa.date32())

# Partition pruning: only one directory's fragments match
fragments = list(dataset.get_fragments(filter=day_filter))
print("fragments after partition pruning:", len(fragments), "of", len(list(dataset.get_fragments())))

# Row-group pruning inside the surviving file: a narrow time window touches few row groups
window_start = datetime.combine(target_day, datetime.min.time(), timezone.utc) + timedelta(hours=1)
ts_type = pa.timestamp("us", tz="UTC")
ts_window = (ds.field("order_ts") >= pa.scalar(window_start, ts_type)) & (
    ds.field("order_ts") < pa.scalar(window_start + timedelta(minutes=30), ts_type)
)
window = day_filter & ts_window
fragment = fragments[0]
kept = fragment.split_by_row_group(filter=ts_window)
print("row groups kept:", len(kept), "of", fragment.metadata.num_row_groups)

result = dataset.to_table(columns=["order_id", "amount"], filter=window)
print("matching rows:", result.num_rows)

assert len(fragments) == 1
assert len(kept) < fragment.metadata.num_row_groups
assert result.num_rows == 30 * 60 // 8

The `explain()` output is the interview evidence: look for `PushedFilters` and `PartitionFilters` rather than a full scan.

```text
filtered = spark.read.parquet("s3://acme-curated/orders/").filter("order_date = '2026-10-04' AND country = 'DE'")
filtered.explain("formatted")
  PartitionFilters: [isnotnull(order_date#..), (order_date#.. = 2026-10-04)]
  PushedFilters:    [IsNotNull(country), EqualTo(country,DE)]
```

The statistics an engine reads to skip row groups:

In [ ]:
pf = pq.ParquetFile(fragment.path)
ts_index = pf.schema_arrow.get_field_index("order_ts")
for rg in range(pf.metadata.num_row_groups)[:3]:
    stats = pf.metadata.row_group(rg).column(ts_index).statistics
    print(f"row group {rg}: rows={pf.metadata.row_group(rg).num_rows} min={stats.min} max={stats.max} nulls={stats.null_count}")
assert pf.metadata.row_group(0).column(ts_index).statistics.has_min_max

Two design levers make skipping effective: **partition on the columns you filter most**, and **sort or cluster within files on the next most selective column** so that min/max ranges are tight. If a file mixes all dates, no row group can be skipped.

Common mistake: writing a filter that cannot be pushed down. A cast such as `WHERE CAST(order_ts AS DATE) = ...`, a function on the column, or a `LIKE '%x'` hides the raw column from the reader, so statistics cannot prune anything. Filter on the raw column and let the engine handle formatting. Also confirm that `write_statistics` was not turned off, because a file written without statistics can never be skipped.

## What are row groups, file sizes, and the small files problem?

A Parquet file is split into row groups (commonly 128 MB by default in Spark), and each row group contains one column chunk per column. Size choices trade off:

+ Larger row groups: better compression and fewer reads, but coarser skipping and more memory per read.
+ Smaller row groups: finer skipping, more overhead and worse compression.

The bigger production problem is file count. Thousands of tiny files per partition hurt every engine: each file means one open, one footer parse, and one task.

Small-file math: a daily partition of 100 GB written as 1 MB files is about 100,000 files. Each file costs one object-store open, one footer parse, and one task, so the engine spends more time on metadata than on data. Compacted to 128 MB files the same partition is about 800 files. A common rule is to keep files between 128 MB and 1 GB and to keep the total file count per table in the tens of thousands, not millions.

```text
bad layout:  s3://acme-curated/orders/ds=2026-10-04/part-00000 ... part-04800  (KB-sized files)
good layout: s3://acme-curated/orders/ds=2026-10-04/part-00000 ... part-00007  (~128-512 MB files)
```

Compaction is read-then-rewrite. The cell creates 40 tiny files, then compacts them into one file with the same rows:

In [ ]:
small_dir = WORKDIR / "small_files"
small_dir.mkdir()
for i, batch in enumerate(orders.to_batches(max_chunksize=1_000)):
    pq.write_table(pa.Table.from_batches([batch]), small_dir / f"part-{i:05d}.parquet")
small_files = sorted(small_dir.glob("*.parquet"))

compacted_path = WORKDIR / "compacted.parquet"
pq.write_table(ds.dataset(small_dir).to_table(), compacted_path, row_group_size=20_000)

small_total = sum(p.stat().st_size for p in small_files)
print(f"{len(small_files)} small files, {small_total:,} bytes -> 1 file, {compacted_path.stat().st_size:,} bytes")
assert len(small_files) == 40
assert pq.ParquetFile(compacted_path).metadata.num_rows == ROWS
assert compacted_path.stat().st_size < small_total  # 40 footers and worse compression disappear

```bash
# Find the small-file problem before it becomes an incident
aws s3api list-objects-v2 --bucket acme-curated --prefix orders/ds=2026-10-04/ \
  --query 'length(Contents)' --output text
hdfs dfs -count -v hdfs:///warehouse/orders/ds=2026-10-04/
```

Common mistake: compacting with `coalesce` and calling it done. `coalesce` reduces files without a shuffle, but it can produce uneven file sizes and it rewrites the whole partition. If the job runs daily on live data, use the table format's maintenance (`OPTIMIZE`, `rewrite_data_files`, `compact`) so commits stay atomic and readers keep a consistent snapshot.

## How does partitioning interact with Parquet?

The partition key becomes a directory level (`orders/ds=2026-10-04/country=DE/part-....parquet`), and the value is also stored in the file for safety. Partitioning is the highest-leverage performance decision and the easiest one to get wrong.

Partitioning is directory-based. The writer removes the partition columns from the file body by default and encodes the values in the path as `key=value` segments. The reader lists directories, parses those values, and injects them as columns. That listing is cheap on a few thousand directories and expensive on hundreds of thousands, which is the main reason over-partitioning hurts.

+ Partition on low-to-medium cardinality columns used in filters: date, region, tenant, status.
+ Keep partition count manageable. A few thousand partitions per table is fine; hundreds of thousands is not.
+ Do not partition on high-cardinality keys such as `user_id` or millisecond timestamps - each partition becomes a tiny file.
+ Prefer one date grain: `ds=YYYY-MM-DD` for daily, `hour=` only when volume truly needs it.
+ For low-cardinality columns (like `country`), consider clustering or sorting inside files instead of an extra directory level.

Worked example: three years of daily orders is about 1,095 partitions. If each day is 1 GB, a 256 MB target gives about 4 files per day, so about 4,380 files for the table. Add `country` with 40 values and the same table becomes about 175,000 directories, most holding a few megabytes. That is the over-partitioning cliff: more directories, smaller files, and a larger metadata bill.

```python
# Over-partitioned: one directory per customer, tiny files, slow metadata operations
orders.write.partitionBy("order_date", "hour", "customer_id").parquet("s3://acme-bad/orders/")

# Right-sized: day grain, country clustered inside files
(
    orders.repartition("order_date")
    .sortWithinPartitions("country", "order_id")
    .write.partitionBy("order_date")
    .parquet("s3://acme-good/orders/")
)
```

Hidden partitioning in Iceberg removes most of this manual reasoning: you declare `days(ts)` or `bucket(16, user_id)` and the engine handles pruning without exposing directory columns in queries.

## What compression and encoding should you use?

Parquet compresses per column chunk and chooses encodings from the data type (dictionary encoding for repeated strings, run-length encoding for sorted values, delta encodings for timestamps). Codec choice is the knob you control:

Encoding and compression are separate stages. The writer first encodes values (dictionary, RLE, delta, bit-packing), then compresses the encoded bytes with the codec. Dictionary encoding only wins while the distinct values fit in the dictionary page (`dictionary_pagesize_limit`, 1 MiB by default); past that the column falls back to plain encoding and re-encoding costs CPU. A column of unique UUIDs compresses poorly; a column of 40 country codes compresses extremely well.

| Codec | Ratio | Speed | Splittable | Typical use |
|---|---|---|---|---|
| Snappy | moderate | fast | yes | Spark default; balanced hot path |
| Zstd | high | medium | yes | cold storage, cost-sensitive lakes |
| Gzip | high | slow | yes | legacy; splits at row groups, but decompression is CPU-heavy |
| LZ4 | low-moderate | fastest | yes | latency-critical reads |
| None | 1x | fastest CPU | yes | already compressed payloads |

In [ ]:
codec_sizes = {}
for codec in ["none", "snappy", "zstd", "gzip"]:
    path = WORKDIR / f"orders_{codec}.parquet"
    pq.write_table(orders, path, compression=codec)
    codec_sizes[codec] = path.stat().st_size
    print(f"{codec:7} {codec_sizes[codec]:>10,} bytes")

# Dictionary encoding: a 5-value column compresses far better than a unique one
dict_path = WORKDIR / "dict.parquet"
pq.write_table(orders.select(["country", "order_id"]), dict_path, compression="none")
rg = pq.ParquetFile(dict_path).metadata.row_group(0)
print("country:", rg.column(0).total_compressed_size, "bytes", rg.column(0).encodings)
print("order_id:", rg.column(1).total_compressed_size, "bytes", rg.column(1).encodings)

assert codec_sizes["zstd"] < codec_sizes["snappy"] < codec_sizes["none"]
assert rg.column(0).total_compressed_size < rg.column(1).total_compressed_size / 5

```python
orders.write.option("compression", "zstd").parquet("s3://acme-cold/orders/")   # per write
spark.conf.set("spark.sql.parquet.compression.codec", "zstd")                   # per session
```

Also reduce the data itself: cast wide strings, drop unused columns before writing, and avoid storing raw JSON blobs as string columns. Encoding cannot fix a schema that stores 2 KB of unused JSON per row.

Common mistake: choosing a codec by ratio alone. Zstd at a high level costs CPU on every read of a hot table, and Gzip is slower still. Snappy or LZ4 usually wins for interactive queries, while Zstd wins for cold data and storage-billed lakes. Measure on your own data, because the ratio depends on cardinality, type, and sort order, not just on the codec name.

## How do nested types work in Parquet?

Nested values are flattened into leaf columns using repetition and definition levels, the model Parquet inherited from Dremel. A `struct` becomes one column chunk per leaf field; a `list<struct>` becomes a repeated group whose children are separate chunks. The levels record how many list elements and how many optional fields are present, so the reader can rebuild rows without storing offsets in the leaf data. Projecting `customer.country` therefore reads only that leaf chunk.

+ Read only the nested leaves you need with dot notation (`customer.country`) so the engine can skip the other sub-columns.
+ Do not `explode` large arrays earlier than necessary; exploding multiplies rows and destroys the columnar advantage.

In [ ]:
nested_schema = pa.schema([
    ("event_id", pa.string()),
    ("customer", pa.struct([("customer_id", pa.int64()), ("country", pa.string()), ("tier", pa.string())])),
    ("items", pa.list_(pa.struct([("sku", pa.string()), ("qty", pa.int32()), ("price", pa.decimal128(10, 2))]))),
])
nested_rows = [
    {
        "event_id": f"e{i}",
        "customer": {"customer_id": i, "country": COUNTRIES[i % 5], "tier": "gold" if i % 3 == 0 else "basic"},
        "items": [{"sku": f"sku-{i}-{k}", "qty": k + 1, "price": Decimal("9.99")} for k in range(i % 3 + 1)],
    }
    for i in range(1_000)
]
events_path = WORKDIR / "events.parquet"
pq.write_table(pa.Table.from_pylist(nested_rows, schema=nested_schema), events_path, compression="zstd")

row_group = pq.ParquetFile(events_path).metadata.row_group(0)
leaves = [row_group.column(i).path_in_schema for i in range(row_group.num_columns)]
print(leaves)

# Project one nested leaf: only that column chunk is read
countries = pq.read_table(events_path, columns=["customer.country"])
print(countries.schema)

assert leaves == [
    "event_id", "customer.customer_id", "customer.country", "customer.tier",
    "items.list.element.sku", "items.list.element.qty", "items.list.element.price",
]

## How does schema evolution work with Parquet?

Schema evolution works because the schema is stored per file, not per dataset. The reader reads each file's footer, aligns columns by name, and fills missing columns with null. Adding a nullable column is safe: old files simply have no chunk for it. Widening an integer type is safe if the reader can upcast. Removing, renaming, or incompatibly retyping a column is not evolution, because old and new files disagree about meaning, not just about presence.

In [ ]:
evolve_dir = WORKDIR / "events_evolving"
evolve_dir.mkdir()
old = pa.table({"event_id": ["a", "b"], "amount": pa.array([10, 20], pa.int32())})
new = pa.table({"event_id": ["c"], "amount": pa.array([30], pa.int64()), "channel": ["web"]})
pq.write_table(old, evolve_dir / "ds=2026-10-01.parquet")
pq.write_table(new, evolve_dir / "ds=2026-10-04.parquet")

merged = pa.unify_schemas(
    [pq.read_schema(evolve_dir / "ds=2026-10-01.parquet"), pq.read_schema(evolve_dir / "ds=2026-10-04.parquet")],
    promote_options="permissive",  # allows int32 -> int64 widening
)
print(merged)

# Pin the merged schema so every file is reconciled the same way; old files get nulls for `channel`
table = ds.dataset(evolve_dir, schema=merged).to_table().sort_by("event_id")
print(table.to_pylist())

assert table.column("channel").to_pylist() == [None, None, "web"]
assert table.schema.field("amount").type == pa.int64()

The Spark equivalent:

```python
events = spark.read.option("mergeSchema", "true").parquet("s3://acme-curated/events/")
```

Rules that avoid incidents:

+ Add columns as nullable; never reuse a column name with a new meaning.
+ `mergeSchema` on every read slows planning for wide schemas; prefer catalog-driven tables (Iceberg/Delta/Hive metastore) over schema inference in production.
+ Renames are migrations, not evolutions. Write the new name and backfill, or use a view that maps old to new.
+ Parquet has no schema registry. If producers change types freely, add Avro + a registry in the ingestion path, or enforce a contract in the table format.

## What are common Parquet mistakes in production?

+ **Writing thousands of small files.** Use fewer, larger files and schedule compaction.
+ **Over-partitioning.** Every extra partition column multiplies directories and shrinks files.
+ **Partitioning by a column you never filter on.** You pay metadata cost and get no pruning.
+ **Non-splittable codecs.** A single gzip CSV cannot be split; gzip Parquet reads can still bottleneck.
+ **No sort/clustering inside files.** Wide min/max ranges make row-group skipping useless.
+ **Blind `append`.** Retries and backfills duplicate rows; make loads idempotent or use a merge.
+ **Storing everything as string.** You lose statistics quality, encoding, and type safety.
+ **Using Parquet for upserts.** Use Iceberg, Delta, or Hudi when records change.
+ **Forgetting timezone semantics.** Store timestamps consistently (UTC) and document it; naive timestamps across engines cause off-by-hours bugs.
+ **Reading with schema inference in production.** It is slow and can silently change types; pass an explicit schema.

Most Parquet incidents reduce to four numbers: file count, file size, row-group size, and sort order. Check the number of files per partition, the median file size, the number of row groups per file, and whether the column you filter on is sorted inside the file. A healthy table has hundreds of files of 128 MB to 1 GB, a handful of row groups each, and tight min/max ranges on the filter columns.

## How do you inspect Parquet metadata and file layout?

In [ ]:
pf = pq.ParquetFile(explicit_path)
metadata = pf.metadata
print("created by:", metadata.created_by)
print("format version:", metadata.format_version)
print("rows:", metadata.num_rows, "row groups:", metadata.num_row_groups, "footer bytes:", metadata.serialized_size)

for rg in range(metadata.num_row_groups):
    row_group = metadata.row_group(rg)
    print(f"row group {rg}: {row_group.num_rows} rows, {row_group.total_byte_size:,} bytes")
    for c in range(row_group.num_columns):
        column = row_group.column(c)
        stats = column.statistics  # can be None: always guard
        print(f"  {column.path_in_schema:12} {column.compression:5} "
              f"min={getattr(stats, 'min', None)} max={getattr(stats, 'max', None)}")
assert metadata.num_rows == ROWS

```bash
aws s3 ls s3://acme-curated/orders/ds=2026-10-04/ --recursive --human-readable --summarize
python -c "import pyarrow.parquet as pq; print(pq.ParquetFile('part-00000.snappy.parquet').metadata)"
```

Common mistake: assuming statistics are always present. Small row groups, all-null columns, and writers configured with `write_statistics=False` produce `None` statistics, so code that reads `stats.min` without a guard crashes or reports nonsense. Also note that `pyarrow.parquet.ParquetDataset` is deprecated in favor of `pyarrow.dataset.dataset`, which supports the same partitioned layout plus filtering and projection.

## Parquet vs ORC vs Avro: when do you choose which?

| Format | Orientation | Strengths | Best for | Watch out for |
|---|---|---|---|---|
| Parquet | columnar | ecosystem, nested types, predicate pushdown | lakehouse and Spark/Trino/Athena analytics | no row-level updates on its own |
| ORC | columnar | Hive integration, ACID in Hive, strong compression | Hive-centric warehouses | smaller ecosystem outside Hive |
| Avro | row | compact, schema registry, fast row writes | Kafka topics, landing/streaming, record-at-a-time | poor for analytical scans |
| JSON/CSV | row text | debuggable, universal | raw landing, small inputs | size, parsing cost, no types |

The formats differ in the unit that gets compressed and in how rows are reassembled. Parquet and ORC are both columnar with row-group-like stripes, but Parquet has the broader ecosystem and stronger nested-type support across Spark, Trino, DuckDB, BigQuery, and Athena; ORC is strongest inside Hive and has built-in Hive ACID support. Avro is row-oriented with compact binary encoding and a schema registry story, which is why it is common on Kafka topics. None of them provides transactions on its own: Delta, Iceberg, and Hudi add a metadata log, ACID commits, snapshots, and time travel on top of Parquet files.

## How do you tune Parquet files for performance?

Parquet tuning is four decisions in order: how many files, how big each file is, how the data is ordered inside each file, and how it is encoded and compressed. Get the first three right and the format does the work; tune codecs only after the layout is correct. Every engine exposes the same knobs under different names, so the reasoning transfers from Spark to Trino, DuckDB, Athena, and Snowflake unloads.

Aim for row groups of 64-128 MB, which is usually a few hundred thousand to a couple of million rows depending on width. Row groups are the unit of statistics skipping and of intra-file parallelism, so too few row groups means one task reads a whole file, and too many means per-row-group overhead and weaker compression. In pyarrow this is `row_group_size` in rows; in Spark it is `spark.sql.parquet.rowGroupSize` in bytes.

```sql
-- Spark: size the write for the target file count
SET spark.sql.files.maxPartitionBytes = 536870912;   -- 512 MB per task
SET spark.sql.parquet.compression.codec = zstd;
SET spark.sql.parquet.rowGroupSize = 134217728;      -- 128 MB row groups

-- Delta Lake: compact small files and order rows inside files
OPTIMIZE analytics.orders ZORDER BY (country, order_ts);

-- Iceberg: rewrite data files and manifests, then drop old snapshots
CALL catalog.system.rewrite_data_files(table => 'analytics.orders');
CALL catalog.system.rewrite_manifests('analytics.orders');
CALL catalog.system.expire_snapshots('analytics.orders', TIMESTAMP '2026-10-01 00:00:00');
```

Compression trades storage and I/O against CPU. Snappy and LZ4 keep CPU low for hot, interactive reads; Zstd level 1-3 gives a better ratio at moderate cost; Gzip and high Zstd levels belong on cold data written once and read rarely. Before tuning the codec, sort or cluster the data on the most selective filter column: tight min/max ranges let the reader skip whole row groups, which saves far more I/O than a better codec.

A short checklist:
+ Write statistics (`write_statistics=True`) and always read with column projection.
+ Sort or bucket within files on the column you filter most, so min/max ranges are tight.
+ Keep row groups per file in the tens, not the thousands.
+ Compact small files on a schedule, and use the table format's maintenance instead of hand-rewriting live partitions.
+ Re-measure after each change: bytes scanned, files read, and plan time are the numbers that move.

## Real world use cases for Parquet

### 1. Event lake on S3 queried by Athena and Spark

Clickstream JSON is converted daily to `ds=YYYY-MM-DD` partitioned Parquet with only the columns analysts query. Athena bills by bytes scanned, so projection plus partition pruning is a direct cost lever.

### 2. Offloading a warehouse to object storage

A fintech exports large cold fact tables from Snowflake to Parquet monthly, keeping only recent months in the warehouse. Historical analysis runs on Spark or Trino against S3 at a fraction of the cost.

```sql
-- Snowflake: unload one month as partitioned Parquet
COPY INTO @acme_stage/orders/month=2026-09/
FROM analytics.orders
WHERE order_date >= '2026-09-01' AND order_date < '2026-10-01'
FILE_FORMAT = (TYPE = PARQUET COMPRESSION = ZSTD)
PARTITION_BY = (order_date)
HEADER = TRUE;
```

Why Parquet: it is portable across engines, so the cold copy is not locked into one vendor.

### 3. Training snapshots for machine learning

A feature platform materializes a daily training snapshot as Parquet. Immutable snapshots make experiments reproducible, and projection keeps reads fast with hundreds of features.

### 4. CDC landing zone feeding a lakehouse table

Debezium writes change events to Kafka; a sink writes micro-batch Parquet into a landing prefix; a merge job applies changes to an Iceberg table. Parquet is the transport format, Iceberg is the queryable table.

Why this split: Parquet is cheap to write and engine-neutral, while the table format provides ACID merges, snapshot isolation, and time travel.

### 5. Log analytics with selective columns

A SaaS platform stores 5 TB/day of application logs. Only a few columns are queried regularly, so the pipeline writes Parquet with a compact set of typed columns plus a compressed `payload` string, and separates high-volume debug logs into a Zstd-compressed cold prefix.

## Parquet exercises

These exercises model Parquet layout arithmetic with the standard library. The numbers are estimates: real sizes depend on types, cardinality, sort order, and codec. The point is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Estimate the file count for a dataset

**Task.** For a 2 TiB dataset, compute how many files you get at target sizes of 128 MB, 512 MB, and 1 GB.

**Hint.** `files = ceil(total_bytes / target_bytes)`; use `math.ceil`.

**Expected result.** 16,384 files at 128 MB, 4,096 at 512 MB, and 2,048 at 1 GB.

In [ ]:
import math

TOTAL_BYTES = 2 * 1024**4   # 2 TiB
counts = {mb: math.ceil(TOTAL_BYTES / (mb * 1024**2)) for mb in [128, 512, 1024]}
for mb, n in counts.items():
    print(f"target {mb:>4} MB -> {n:>6,} files")

assert counts[128] == 16_384
assert counts[512] == 4_096
assert counts[1024] == 2_048

### Try it 2. Compression ratio and storage math

**Task.** 200 GB of JSON becomes 45 GB as Snappy Parquet and 35 GB as Zstd Parquet. Compute the compression ratio of each and the monthly storage cost of all three at 0.02 per GB-month.

**Hint.** `ratio = original / stored` and `cost = stored_gb * price`.

**Expected result.** Snappy is 4.44x and Zstd is 5.71x; Zstd costs 0.70 per month against 4.00 for raw JSON, saving 3.30.

In [ ]:
original_gb = 200
snappy_gb = 45
zstd_gb = 35
price_per_gb_month = 0.02

for name, gb in [("json", original_gb), ("snappy", snappy_gb), ("zstd", zstd_gb)]:
    ratio = original_gb / gb
    cost = gb * price_per_gb_month
    print(f"{name:7} {gb:>4} GB  ratio={ratio:5.2f}x  cost=${cost:5.2f}/month")

savings = original_gb * price_per_gb_month - zstd_gb * price_per_gb_month
print(f"zstd saves ${savings:.2f} per month vs raw JSON")

assert round(original_gb / snappy_gb, 2) == 4.44
assert round(original_gb / zstd_gb, 2) == 5.71
assert round(savings, 2) == 3.30

### Try it 3. Row-group count versus threads

**Task.** A 4 GB file is read by 16 threads. Compute the number of row groups for row-group sizes of 64 MB, 128 MB, and 512 MB, and say whether the file is split finely enough to keep every thread busy.

**Hint.** `row_groups = ceil(file_mb / row_group_mb)`; enough means at least one row group per thread.

**Expected result.** 64 and 128 MB give 64 and 32 row groups (enough); 512 MB gives only 8 row groups, so half the threads idle.

In [ ]:
import math

FILE_MB = 4096
THREADS = 16

for rg_mb in [64, 128, 512]:
    groups = math.ceil(FILE_MB / rg_mb)
    per_thread = groups / THREADS
    verdict = "enough" if groups >= THREADS else "too coarse"
    print(f"row_group={rg_mb:>4} MB -> {groups:>3} row groups, "
          f"{per_thread:.2f} per thread, {verdict}")

assert math.ceil(FILE_MB / 64) == 64
assert math.ceil(FILE_MB / 128) == 32
assert math.ceil(FILE_MB / 512) == 8

### Try it 4. Dictionary-encoding break-even

**Task.** A column has 1,000,000 rows, 24 bytes per value, and a 4-byte dictionary index. Model bytes as `distinct * value_bytes + rows * index_bytes` for dictionary encoding and `rows * value_bytes` for plain. Compute both at 1,000 and at 1,000,000 distinct values, and find the break-even distinct count.

**Hint.** Dictionary encoding wins while `distinct * (value_bytes - index_bytes) < rows * value_bytes`; solve for `distinct`.

**Expected result.** Dictionary wins at 1,000 distinct values and loses at 1,000,000; break-even is about 833,333 distinct values.

In [ ]:
VALUE_BYTES = 24
INDEX_BYTES = 4
ROWS = 1_000_000


def dictionary_bytes(distinct):
    return distinct * VALUE_BYTES + ROWS * INDEX_BYTES


def plain_bytes():
    return ROWS * VALUE_BYTES


print(f"plain = {plain_bytes():,} bytes")
for distinct in [1_000, 100_000, 1_000_000]:
    d = dictionary_bytes(distinct)
    print(f"distinct={distinct:>9,} dict={d:>12,} wins={d < plain_bytes()}")

break_even = (VALUE_BYTES - INDEX_BYTES) * ROWS / VALUE_BYTES
print(f"break-even distinct values ~= {break_even:,.0f}")

assert dictionary_bytes(1_000) < plain_bytes()
assert dictionary_bytes(1_000_000) > plain_bytes()
assert round(break_even) == 833_333

### Try it 5. Cost of reading one column versus all

**Task.** A file has 40 equal-sized columns and is 800 MB in total. Compute the bytes read when a query selects 3 columns, 5 columns, and all 40, and the percentage of the file each represents.

**Hint.** `bytes_read = selected_columns * (total_mb / column_count)`.

**Expected result.** 3 columns read 60 MB (7.5%), 5 columns read 100 MB (12.5%), and all 40 read 800 MB.

In [ ]:
TOTAL_MB = 800
COLUMNS = 40
COLUMN_MB = TOTAL_MB / COLUMNS


def scan(selected):
    return selected * COLUMN_MB


for n in [3, 5, 40]:
    mb = scan(n)
    print(f"{n:>2} columns -> {mb:6.1f} MB ({mb / TOTAL_MB * 100:5.1f}% of the file)")

assert round(scan(3), 1) == 60.0
assert round(scan(3) / TOTAL_MB * 100, 1) == 7.5
assert round(scan(40), 1) == 800.0

### Try it 6. Small-file compaction savings

**Task.** Each file costs 5 ms of open and footer-parse time. Compare 100,000 files of 1 MB with the same data compacted into 800 files of about 125 MB. Compute the metadata time before and after and the saving.

**Hint.** `seconds = files * overhead_ms / 1000`.

**Expected result.** 500 s before, 4 s after, a saving of 496 s.

In [ ]:
OVERHEAD_MS = 5
before_files = 100_000
after_files = 800

before = before_files * OVERHEAD_MS / 1000
after = after_files * OVERHEAD_MS / 1000
saved = before - after
print(f"metadata time: before {before:,.0f}s, after {after:.0f}s, saved {saved:,.0f}s")

assert before == 500.0
assert after == 4.0
assert saved == 496.0

### Try it 7. Choose a codec by CPU and storage tradeoff

**Task.** For 1,000 GB of raw data scanned 10 times a month, compute the monthly cost of four codecs. Storage costs 0.023 per GB-month and CPU costs 0.001 per GB scanned for each unit of relative CPU. Total cost is `stored_gb * storage_price + stored_gb * scans * cpu_factor * cpu_price`. Pick the cheapest.

**Hint.** `stored_gb = raw_gb * ratio`, then `min(totals, key=totals.get)`.

**Expected result.** Zstd is cheapest at about 7.38 per month; Gzip loses to CPU at 9.01, Snappy is 8.25, and no compression is 25.00 on storage.

In [ ]:
STORAGE_PRICE = 0.023      # per GB-month
SCANS = 10                 # full scans per month
CPU_PRICE = 0.001          # per GB-scan of relative CPU
RAW_GB = 1000

# codec -> (compression ratio, relative CPU per byte scanned)
codecs = {
    "none":   (1.00, 0.2),
    "snappy": (0.25, 1.0),
    "zstd":   (0.18, 1.8),
    "gzip":   (0.17, 3.0),
}

totals = {}
for name, (ratio, cpu) in codecs.items():
    stored = RAW_GB * ratio
    total = stored * STORAGE_PRICE + stored * SCANS * cpu * CPU_PRICE
    totals[name] = total
    print(f"{name:7} stored={stored:6.0f} GB  monthly cost=${total:6.2f}")

cheapest = min(totals, key=totals.get)
print("cheapest:", cheapest)

assert cheapest == "zstd"
assert round(totals["zstd"], 2) == 7.38
assert round(totals["gzip"], 2) == 9.01
assert round(totals["none"], 2) == 25.00

### Try it 8. Row-group skipping with min/max statistics

**Task.** A sorted file has 200 row groups of 1,000 rows each. Row group `i` holds values from `i * 1000` to `i * 1000 + 999`. A filter matches values from 80,000 to 119,999. Count how many row groups are read and how many are skipped by min/max statistics.

**Hint.** Skip a row group when `group_max < low` or `group_min > high`.

**Expected result.** 40 row groups are read and 160 are skipped, an 80% skip rate.

In [ ]:
GROUPS = 200
GROUP_SIZE = 1000
LOW = 80 * GROUP_SIZE
HIGH = 119 * GROUP_SIZE + (GROUP_SIZE - 1)

read = 0
skipped = 0
for i in range(GROUPS):
    group_min = i * GROUP_SIZE
    group_max = group_min + GROUP_SIZE - 1
    if group_max < LOW or group_min > HIGH:
        skipped += 1
    else:
        read += 1

print(f"read {read} of {GROUPS} row groups, skipped {skipped} "
      f"({skipped / GROUPS * 100:.0f}%)")

assert read == 40
assert skipped == 160
assert skipped / GROUPS == 0.8